In [1]:
import sqlite3
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from datetime import datetime

# ---- Reconstruct feature columns exactly as used in training ----
df = pd.read_csv('data/processed/clean_data.csv')
exclude_cols = ['unit_number', 'time_in_cycles', 'RUL']
feature_cols = [col for col in df.columns if col not in exclude_cols]
print("Feature columns:", feature_cols)
print("Total rows loaded:", len(df))

Feature columns: ['op_setting_1', 'op_setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_20', 'sensor_21', 'sensor_2_roll_mean_5', 'sensor_2_roll_std_5']
Total rows loaded: 20631


In [2]:
# ---- Load XGBoost model ----
xgb_model = XGBClassifier()
xgb_model.load_model('models/xgboost_model.json')
print("XGBoost model loaded.")

# ---- Load PyTorch model ----
class RULPredictor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.network(x)

input_dim = len(feature_cols)
pytorch_model = RULPredictor(input_dim)
pytorch_model.load_state_dict(torch.load('models/pytorch_model.pth'))
pytorch_model.eval()
print("PyTorch model loaded.")

XGBoost model loaded.
PyTorch model loaded.


In [3]:
# ---- Refit scaler on the same data distribution used in training ----
# (In production this would be saved from training; here we refit on full clean data
# since PyTorch needs scaled inputs to predict correctly)
reg_scaler = StandardScaler()
reg_scaler.fit(df[feature_cols])
print("Scaler fitted on", len(df), "rows.")

# ---- Create the SQL database and table ----
conn = sqlite3.connect('data/engine_logbook.db')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS predictions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    machine_id INTEGER NOT NULL,
    cycle INTEGER NOT NULL,
    status TEXT NOT NULL,
    confidence REAL NOT NULL,
    predicted_rul REAL,
    timestamp TEXT NOT NULL
)
''')
conn.commit()
print("Database and table ready at data/engine_logbook.db")

Scaler fitted on 20631 rows.
Database and table ready at data/engine_logbook.db


In [4]:
def predict_and_log(machine_id, cycle_data_row, cursor, conn):
    """
    Takes one row of sensor data, runs it through both models,
    and writes a new entry to the database - exactly like your
    original Step 4 plan described.
    """
    # Prepare the row as a DataFrame with correct feature order
    X_row = pd.DataFrame([cycle_data_row[feature_cols]], columns=feature_cols)

    # --- XGBoost: classify failing soon or not ---
    xgb_pred = xgb_model.predict(X_row)[0]
    xgb_confidence = xgb_model.predict_proba(X_row)[0][1]  # confidence of "failing"
    status = "Failing" if xgb_pred == 1 else "Healthy"

    # --- PyTorch: predict remaining cycles ---
    X_scaled = reg_scaler.transform(X_row)
    X_tensor = torch.FloatTensor(X_scaled)
    with torch.no_grad():
        predicted_rul = pytorch_model(X_tensor).item()

    # --- Write to database ---
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cursor.execute('''
        INSERT INTO predictions (machine_id, cycle, status, confidence, predicted_rul, timestamp)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (int(machine_id), int(cycle_data_row['time_in_cycles']), status,
          float(xgb_confidence), float(predicted_rul), timestamp))
    conn.commit()

    return status, xgb_confidence, predicted_rul

# Test it on one real row
sample_row = df[df['unit_number'] == 1].iloc[-1]  # last recorded cycle of engine 1
status, conf, rul = predict_and_log(1, sample_row, cursor, conn)
print(f"Machine ID: 1 | Status: {status} | Confidence: {conf:.2%} | Predicted RUL: {rul:.1f} cycles")

Machine ID: 1 | Status: Failing | Confidence: 99.99% | Predicted RUL: 17.8 cycles


In [5]:
# Log the latest cycle for every engine - populates the full logbook
for unit in df['unit_number'].unique():
    latest_row = df[df['unit_number'] == unit].iloc[-1]
    predict_and_log(unit, latest_row, cursor, conn)

print("Logged predictions for all 100 engines.")

# Verify - read back from the database
result = pd.read_sql_query("SELECT * FROM predictions ORDER BY id DESC LIMIT 10", conn)
print(result)

Logged predictions for all 100 engines.
     id  machine_id  cycle   status  confidence  predicted_rul  \
0  2615         100    200  Failing    0.999865      26.134605   
1  2614          99    185  Failing    0.999958      21.954071   
2  2613          98    156  Failing    0.999966      53.013950   
3  2612          97    202  Failing    0.999964       9.077475   
4  2611          96    336  Failing    0.999983      20.716843   
5  2610          95    283  Failing    0.999985       6.999446   
6  2609          94    258  Failing    0.999990       8.405433   
7  2608          93    155  Failing    0.999910      28.586498   
8  2607          92    341  Failing    0.999972       9.080876   
9  2606          91    135  Failing    0.999918      20.752134   

             timestamp  
0  2026-10-03 17:22:36  
1  2026-10-03 17:22:36  
2  2026-10-03 17:22:36  
3  2026-10-03 17:22:36  
4  2026-10-03 17:22:36  
5  2026-10-03 17:22:36  
6  2026-10-03 17:22:36  
7  2026-10-03 17:22:36  
8  2026-

In [6]:
# More realistic demo: log a random EARLY cycle for some engines, LATE cycle for others
# This simulates monitoring engines at different points in their operational life
import random
random.seed(42)

cursor.execute("DELETE FROM predictions")  # clear the "all failing" test data
conn.commit()

for unit in df['unit_number'].unique():
    engine_data = df[df['unit_number'] == unit]
    max_cycle_idx = len(engine_data) - 1

    # Pick 3 checkpoints per engine: early (20%), mid (60%), and late (95%) life
    checkpoints = [int(max_cycle_idx * 0.2), int(max_cycle_idx * 0.6), int(max_cycle_idx * 0.95)]

    for idx in checkpoints:
        row = engine_data.iloc[idx]
        predict_and_log(unit, row, cursor, conn)

print("Logged 3 checkpoints (early/mid/late life) for all 100 engines.")

# Check the status distribution now
result = pd.read_sql_query("SELECT status, COUNT(*) as count FROM predictions GROUP BY status", conn)
print(result)

Logged 3 checkpoints (early/mid/late life) for all 100 engines.
    status  count
0  Failing    101
1  Healthy    199


In [7]:
# Calculate estimated cost savings: every correctly caught "Failing" prediction
# represents an early warning that could prevent unplanned downtime
COST_SAVINGS_PER_CATCH = 50000  # ₹50,000 per early catch, as per your original plan

result = pd.read_sql_query(
    "SELECT COUNT(*) as failing_catches FROM predictions WHERE status = 'Failing'",
    conn
)
failing_catches = result['failing_catches'][0]
total_savings = failing_catches * COST_SAVINGS_PER_CATCH

print(f"Total 'Failing' predictions caught: {failing_catches}")
print(f"Estimated cost savings: ₹{total_savings:,}")

Total 'Failing' predictions caught: 101
Estimated cost savings: ₹5,050,000
